# Add results from the field

> Match laboratory results to areas, ready for the next round

::: {.callout-note appearance="simple" icon=false}
**How-to guide: get one job done.** It assumes that you know the basics. New to trufl? Take the [tutorial](../tutorials/01_getting_started.ipynb) first. [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

**You need:** `areas`, the earlier `measurements` (the empty table, after a first round), the survey used as prior, `survey_file`, and a table `results` with the columns `lat`, `lng` and `value`.

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

fname_raster = '../files/ground-truth-01-4326-simulated.tif'
areas = gridder(fname_raster)
no_samples = gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None})
prior_state = State(no_samples, areas, cbs=[PriorCB(fname_raster)])()
first_ranks = Optimizer(prior_state).get_rank(is_benefit_x=[True], w_vector=[1], s_method='CP')
first_n = rank_to_sample(first_ranks['rank'].reindex(areas.index).to_numpy(), budget=600, min=1, policy='quantiles')
measurements = DataCollector(fname_raster).collect(
    Sampler(areas).sample(first_n, method='uniform', rng=np.random.default_rng(0)))
state = State(measurements, areas, cbs=[MaxCB(), MinCB(), StdCB(), CountCB(), PriorCB(fname_raster)])()
ranks = Optimizer(state).get_rank(is_benefit_x=[True, True, True, False, True],
                                  w_vector=[0.25, 0.125, 0.125, 0.25, 0.25], n_method='LINEAR1', s_method='CP')

n = rank_to_sample(ranks['rank'].reindex(areas.index).to_numpy(), budget=400, min=1, policy='quantiles')
_new = DataCollector(fname_raster).collect(Sampler(areas).sample(n, method='uniform', rng=np.random.default_rng(1)))
survey_file = fname_raster
results = _new.get_coordinates().rename(columns={'x': 'lng', 'y': 'lat'}).assign(value=_new['value'].values).reset_index(drop=True)

## 1. Make the results comparable

trufl pools every value and checks none of this, so do it first:

- convert to one quantity and unit, and to one sampling depth;
- decay-correct radionuclide activities to one reference date;
- replace results below the detection limit with a value, following one written rule.

## 2. Refuse blanks and repeats

In [ ]:
assert results['value'].notna().all(), 'some results are blank'
assert not results.duplicated().any(), 'some results appear twice'

## 3. Match each result to its area

In [ ]:
import geopandas as gpd
import pandas as pd

points = gpd.GeoDataFrame(results[['value']], geometry=gpd.points_from_xy(results.lng, results.lat), crs=4326)
new = (gpd.sjoin(points.to_crs(areas.crs), areas.reset_index()[['loc_id', 'geometry']], predicate='within')
       .drop(columns='index_right').set_index('loc_id'))

print(len(results) - len(new), 'results lost')
assert len(new) == len(results), 'a point is outside, on a border, or in two overlapping areas'

0 results lost


If that fails, look at those points on a map. Give each a `loc_id` by hand, or leave it out and record why.

## 4. Join them to the earlier measurements

In [ ]:
all_measurements = pd.concat([measurements, new])
print(len(all_measurements), 'measurements')

1000 measurements


## 5. Compute the state of the next round

In [ ]:
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB

state = State(all_measurements, areas, cbs=[MaxCB(), MinCB(), StdCB(), CountCB(), PriorCB(survey_file)])().astype(float)
print(int(state.isna().any(axis=1).sum()), 'areas have a gap')

0 areas have a gap


Settle any gaps as [before the first round](02_use_your_data.ipynb#find-and-settle-the-gaps), then [set your priorities](03_set_priorities.ipynb).